### 1. Database bağlantısının hazırlanması

Bu mərhələdə notebook SQL Server ilə işləməyə hazırlanır. Lazım olan import-lar çağırılır və database connection yaradılır.


In [2]:
from common import *

import common
from db import get_engine

engine = get_engine()

### 1.Qısa nəticə:
Database ilə işləmək üçün connection hazırlandı.

### 2. Silver cədvəlinin yoxlanılması

Bu mərhələdə `silver.flights_clean` cədvəlinin SQL Server-də mövcud olduğu və neçə sətirdən ibarət olduğu yoxlanılır.

In [3]:
import pandas as pd

pd.read_sql("""
SELECT COUNT(*) AS total_rows
FROM silver.flights_clean
""", con=engine)

KeyboardInterrupt: 

### 2.Qısa nəticə:

`silver.flights_clean` cədvəlinin SQL Server-də mövcud olduğu və analiz üçün əlçatan olduğu yoxlanıldı.

## 3. Silver datasının ilkin baxışı 
Bu mərhələdə analiz edəcəyimiz silver.flights_clean cədvəlinin ilk sətirlərinə baxılır. Məqsəd sütunların və məlumatların düzgün göründüyünü yoxlamaqdır.

In [4]:
pd.read_sql("""SELECT TOP 5 *
FROM silver.flights_clean""", con=engine)

,MONTH,DAY,DAY_OF_WEEK,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DISTANCE,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_DELAY,...,WEATHER_SURFACE_PRESSURE,WEATHER_WIND_DIRECTION,WEATHER_SNOWFALL,WEATHER_WIND_GUSTS,YEAR,IS_DELAYED,FLIGHT_STATUS,IS_WEEKEND,DEPARTURE_PERIOD,IS_LONG_DISTANCE
0,June,29,1,EV,ATL,TRI,227,12:05:00,13:08:00,-4.0,...,980.900024,310.0,0.0,14.8,2015,0,Completed,0,Afternoon,0
1,April,19,7,DL,ATL,PNS,271,17:35:00,17:48:00,-2.0,...,975.000000,193.0,0.0,47.5,2015,0,Completed,1,Afternoon,0
2,August,18,2,WN,RDU,STL,667,18:50:00,19:55:00,296.0,...,NaN,NaN,NaN,NaN,2015,1,Completed,0,Evening,0
3,August,16,7,HA,KOA,HNL,163,13:41:00,14:25:00,-3.0,...,NaN,NaN,NaN,NaN,2015,0,Completed,1,Afternoon,0
4,July,30,4,WN,LAX,SJC,308,19:45:00,20:50:00,102.0,...,1010.700012,289.0,0.0,19.1,2015,1,Completed,0,Evening,0


### 4. Ümumi uçuş vəziyyətinin yoxlanılması

Bu mərhələdə uçuşların neçə dənəsinin tamamlandığı, ləğv edildiyi və başqa istiqamətə yönləndirildiyi yoxlanılır.

In [ ]:
pd.read_sql("""
SELECT
    FLIGHT_STATUS,
    COUNT(*) AS FLIGHT_COUNT
FROM silver.flights_clean
GROUP BY FLIGHT_STATUS
ORDER BY FLIGHT_COUNT DESC
""", con=engine)

,FLIGHT_STATUS,FLIGHT_COUNT
0,Completed,981515
1,Cancelled,15527
2,Diverted,2615


### 4.Qısa nəticə:

Analiz nəticəsində uçuşların böyük hissəsinin uğurla tamamlandığı görüldü. Ümumilikdə **981,515 uçuş Completed**, **15,527 uçuş Cancelled**, **2,615 uçuş isə Diverted** olub.

Bu nəticə uçuş əməliyyatlarının ümumi vəziyyətini göstərir və əsas problemlərin daha çox hansı istiqamətlərdə olduğunu araşdırmaq üçün ilkin baza yaradır.


### 5. Aviaşirkətlər üzrə uçuş performansının müqayisəsi

Bu mərhələdə aviaşirkətlərin uçuş performansı müqayisə olunur. Məqsəd hansı aviaşirkətlərdə gecikmə faizinin və orta çatma gecikməsinin daha yüksək və ya aşağı olduğunu görməkdir.

In [6]:
pd.read_sql("""
SELECT
    AIRLINE_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY AIRLINE_NAME
ORDER BY DELAY_RATE
""", con=engine)

,AIRLINE_NAME,TOTAL_FLIGHTS,DELAY_RATE,AVG_ARRIVAL_DELAY
0,Hawaiian Airlines Inc.,13122,10.93,1.87
1,Alaska Airlines Inc.,29609,13.05,-1.06
2,Delta Air Lines Inc.,149858,13.65,0.28
3,American Airlines Inc.,124964,18.20,3.18
4,Skywest Airlines Inc.,100614,18.63,5.83
5,US Airways Inc.,34278,18.89,3.77
6,Southwest Airlines Co.,217546,19.04,4.39
7,Virgin America,10703,19.35,4.91
8,Atlantic Southeast Airlines,98165,19.64,6.66
9,United Air Lines Inc.,88596,20.54,5.33


### 5.Qısa nəticə:

Aviaşirkətlər üzrə müqayisədə ən aşağı gecikmə faizi **Hawaiian Airlines Inc.** şirkətində olub (**10.93%**).

Ən yüksək gecikmə faizi isə **Spirit Air Lines** şirkətində müşahidə olunub (**29.93%**). Spirit Air Lines həmçinin orta hesabla **14.23 dəqiqə** çatma gecikməsi ilə ən zəif nəticələrdən birini göstərib.

**Alaska Airlines Inc.** üçün orta çatma gecikməsinin **-1.06 dəqiqə** olması uçuşların orta hesabla planlaşdırılmış vaxtdan bir qədər tez çatdığını göstərir.

Nəticə olaraq, aviaşirkətlər arasında uçuş performansında nəzərəçarpacaq fərqlər olduğu görünür.
